# Stage 3 — PhoBERT Clean Fine-tuning

This notebook coordinates Stage 3. The full PhoBERT fine-tuning should run on Kaggle GPU, not on the local Intel Arc machine.

Stage 3 goals:

1. Fine-tune PhoBERT-base on clean sentiment classification.
2. Fine-tune PhoBERT-base on clean topic classification.
3. Evaluate on dev and test.
4. Compare PhoBERT against the best Stage 2 baselines.
5. Export all outputs with prefix `03_`.

This notebook does not generate noisy data.


## 1. Local sanity checks

In [2]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "03_phobert_training_config.yaml"
DATA_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
NOTES_DIR = PROJECT_ROOT / "reports" / "notes"

print("Project root:", PROJECT_ROOT)
print("Config exists:", CONFIG_PATH.exists())
print("Train exists:", (DATA_DIR / "train.csv").exists())
print("Dev exists:", (DATA_DIR / "dev.csv").exists())
print("Test exists:", (DATA_DIR / "test.csv").exists())
print("Label mapping exists:", (PROJECT_ROOT / "configs" / "label_mapping.json").exists())


Project root: d:\project-ml-engineering\nlp-phobert-student-feedback
Config exists: True
Train exists: True
Dev exists: True
Test exists: True
Label mapping exists: True


## 2. Inspect Stage 3 config

In [3]:
import yaml

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

config


{'project': {'data_dir': 'data/processed',
  'label_mapping_path': 'configs/label_mapping.json',
  'reports_tables_dir': 'reports/tables',
  'reports_figures_dir': 'reports/figures',
  'reports_notes_dir': 'reports/notes',
  'models_dir': 'models/phobert'},
 'model': {'model_name': 'vinai/phobert-base', 'max_length': 128},
 'preprocessing': {'word_segmentation_enabled': False,
  'word_segmentation_method': 'none'},
 'training': {'seed': 42,
  'learning_rate': 2e-05,
  'num_train_epochs': 4,
  'per_device_train_batch_size': 16,
  'per_device_eval_batch_size': 32,
  'weight_decay': 0.01,
  'warmup_ratio': 0.06,
  'logging_steps': 50,
  'evaluation_strategy': 'epoch',
  'save_strategy': 'epoch',
  'load_best_model_at_end': True,
  'metric_for_best_model': 'macro_f1',
  'greater_is_better': True,
  'fp16': True,
  'report_to': 'none'},
 'tasks': {'sentiment': {'text_column': 'text',
   'label_column': 'sentiment_label',
   'output_subdir': 'sentiment'},
  'topic': {'text_column': 'text',
 

## 3. Inspect processed data and labels

In [4]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
dev_df = pd.read_csv(DATA_DIR / "dev.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

with open(PROJECT_ROOT / "configs" / "label_mapping.json", "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)

print("\nSentiment mapping:")
print(label_mapping["sentiment"])

print("\nTopic mapping:")
print(label_mapping["topic"])

display(train_df.head())


Train: (11426, 9)
Dev: (1583, 9)
Test: (3166, 9)

Sentiment mapping:
{'source_column': 'sentiment', 'verified_from_hf_features': True, 'id_to_name': {'0': 'negative', '1': 'neutral', '2': 'positive'}}

Topic mapping:
{'source_column': 'topic', 'verified_from_hf_features': True, 'id_to_name': {'0': 'lecturer', '1': 'training_program', '2': 'facility', '3': 'others'}}


,id,split,text,sentiment_label_raw,sentiment_label,topic_label_raw,topic_label,char_count,raw_word_count
0,train_0,train,slide giáo trình đầy đủ .,2,positive,1,training_program,25,6
1,train_1,train,"nhiệt tình giảng dạy , gần gũi với sinh viên .",2,positive,0,lecturer,46,11
2,train_2,train,đi học đầy đủ full điểm chuyên cần .,0,negative,1,training_program,36,9
3,train_3,train,chưa áp dụng công nghệ thông tin và các thiết ...,0,negative,0,lecturer,76,18
4,train_4,train,"thầy giảng bài hay , có nhiều bài tập ví dụ ng...",2,positive,0,lecturer,59,15


## 4. Kaggle setup commands

Run these commands inside a Kaggle notebook after enabling GPU.

Use the repository as source of truth. Replace the GitHub URL with your repository URL.


In [5]:
print(r"""
# Kaggle cell 1 — Clone repo
!git clone https://github.com/<your-username>/nlp-phobert-student-feedback.git
%cd nlp-phobert-student-feedback

# Kaggle cell 2 — Install Stage 3 dependencies
!pip install -r requirements_stage3_kaggle.txt

# Kaggle cell 3 — Check GPU
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Kaggle cell 4 — Train sentiment
!python src/train_phobert.py --task sentiment --config configs/03_phobert_training_config.yaml

# Kaggle cell 5 — Train topic
!python src/train_phobert.py --task topic --config configs/03_phobert_training_config.yaml

# Kaggle cell 6 — Package outputs for download
!zip -r /kaggle/working/stage3_phobert_outputs.zip reports/tables/03_* reports/figures/03_* reports/notes/03_* models/phobert
""")



# Kaggle cell 1 — Clone repo
!git clone https://github.com/<your-username>/nlp-phobert-student-feedback.git
%cd nlp-phobert-student-feedback

# Kaggle cell 2 — Install Stage 3 dependencies
!pip install -r requirements_stage3_kaggle.txt

# Kaggle cell 3 — Check GPU
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Kaggle cell 4 — Train sentiment
!python src/train_phobert.py --task sentiment --config configs/03_phobert_training_config.yaml

# Kaggle cell 5 — Train topic
!python src/train_phobert.py --task topic --config configs/03_phobert_training_config.yaml

# Kaggle cell 6 — Package outputs for download
!zip -r /kaggle/working/stage3_phobert_outputs.zip reports/tables/03_* reports/figures/03_* reports/notes/03_* models/phobert



## 5. Optional local dry-run command

Use only for smoke testing on a tiny environment if you have PyTorch installed. Do not use local CPU for full training.


In [6]:
print(r"""
# Optional only. Full training should run on Kaggle GPU.
python src/train_phobert.py --task sentiment --config configs/03_phobert_training_config.yaml
""")



# Optional only. Full training should run on Kaggle GPU.
python src/train_phobert.py --task sentiment --config configs/03_phobert_training_config.yaml



## 6. Load Stage 3 results after copying Kaggle outputs back

After Kaggle finishes, copy these folders back into the local project:

- `reports/tables/03_*`
- `reports/figures/03_*`
- `reports/notes/03_*`
- `models/phobert/` if you need local inference

Then run the cells below.


In [7]:
phobert_summary_path = TABLES_DIR / "03_phobert_results_summary.csv"
comparison_path = TABLES_DIR / "03_phobert_vs_baseline_comparison.csv"

if phobert_summary_path.exists():
    phobert_summary = pd.read_csv(phobert_summary_path)
    display(phobert_summary)
else:
    print("Stage 3 summary not found yet:", phobert_summary_path)

if comparison_path.exists():
    comparison_df = pd.read_csv(comparison_path)
    display(comparison_df)
else:
    print("PhoBERT vs baseline comparison not found yet:", comparison_path)


Stage 3 summary not found yet: d:\project-ml-engineering\nlp-phobert-student-feedback\reports\tables\03_phobert_results_summary.csv
PhoBERT vs baseline comparison not found yet: d:\project-ml-engineering\nlp-phobert-student-feedback\reports\tables\03_phobert_vs_baseline_comparison.csv


## 7. Plot PhoBERT vs baseline comparison locally

In [8]:
import matplotlib.pyplot as plt

if comparison_path.exists():
    comparison_df = pd.read_csv(comparison_path)

    plot_df = comparison_df[[
        "task",
        "baseline_test_macro_f1",
        "phobert_test_macro_f1",
    ]].set_index("task")

    ax = plot_df.plot(kind="bar", figsize=(9, 5))
    ax.set_title("PhoBERT vs Best Stage 2 Baseline - Test Macro-F1")
    ax.set_xlabel("Task")
    ax.set_ylabel("Macro-F1")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=0)
    plt.tight_layout()

    output_path = FIGURES_DIR / "03_phobert_vs_baseline_macro_f1_local_review.png"
    plt.savefig(output_path, dpi=150)
    plt.show()

    print("Saved:", output_path)
else:
    print("Comparison file is not available yet.")


Comparison file is not available yet.


## 8. Final Stage 3 output check

In [9]:
expected_files = [
    TABLES_DIR / "03_phobert_results_summary.csv",
    TABLES_DIR / "03_phobert_classification_reports.csv",
    TABLES_DIR / "03_phobert_sentiment_classification_report.csv",
    TABLES_DIR / "03_phobert_topic_classification_report.csv",
    TABLES_DIR / "03_phobert_vs_baseline_comparison.csv",
    FIGURES_DIR / "03_phobert_vs_baseline_macro_f1.png",
    NOTES_DIR / "03_phobert_clean_finetuning_report.md",
]

print("Stage 3 output check:")

all_ok = True
for path in expected_files:
    exists = path.exists()
    all_ok = all_ok and exists
    print("[OK]     " if exists else "[MISSING]", path)

print("\nStage 3 completed:", all_ok)


Stage 3 output check:
[MISSING] d:\project-ml-engineering\nlp-phobert-student-feedback\reports\tables\03_phobert_results_summary.csv
[MISSING] d:\project-ml-engineering\nlp-phobert-student-feedback\reports\tables\03_phobert_classification_reports.csv
[MISSING] d:\project-ml-engineering\nlp-phobert-student-feedback\reports\tables\03_phobert_sentiment_classification_report.csv
[MISSING] d:\project-ml-engineering\nlp-phobert-student-feedback\reports\tables\03_phobert_topic_classification_report.csv
[MISSING] d:\project-ml-engineering\nlp-phobert-student-feedback\reports\tables\03_phobert_vs_baseline_comparison.csv
[MISSING] d:\project-ml-engineering\nlp-phobert-student-feedback\reports\figures\03_phobert_vs_baseline_macro_f1.png
[MISSING] d:\project-ml-engineering\nlp-phobert-student-feedback\reports\notes\03_phobert_clean_finetuning_report.md

Stage 3 completed: False


## What to send for review

After Stage 3 runs on Kaggle and outputs are copied back, send:

1. `reports/tables/03_phobert_results_summary.csv`
2. `reports/tables/03_phobert_vs_baseline_comparison.csv`
3. `reports/notes/03_phobert_clean_finetuning_report.md`
4. The final output check from this notebook
